# Evaluation

Build an EvalSet to grade tool trajectories.

> Part of the **ADK Katas**. Work top-to-bottom: read the concept, fill in the
> `# TODO`s in the starter cell, then run the **Check** cell — it grades your
> code offline (no API call). The **Run it live** cell needs a `GOOGLE_API_KEY`.


## Kata 09 — Evaluation

ADK can **grade** an agent against an `EvalSet` — recorded cases with the
expected **tool trajectory**. Build cases programmatically from ADK's own
classes so the schema is always valid.

⚠️ Gotcha: `IntermediateData.tool_uses` wants **`types.FunctionCall`**
objects — *not* `types.Part`. Wrapping in a `Part` silently serialises to
`{}` and your trajectory check passes vacuously.

## Your task

Build `eval_set`: an `EvalSet` with one `EvalCase` (`eval_id="weather_tokyo"`)
whose single `Invocation` records a `get_weather(city="Tokyo")` tool call in
`intermediate_data.tool_uses` — using `types.FunctionCall` (not `Part`).

In [ ]:
# Setup — run me first
import sys, pathlib
# Make the kata library importable whether opened from adk-katas/ or adk-katas/notebooks/
for _p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent):
    if (_p / "kata_helpers.py").exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from kata_helpers import *          # load_env, has_api_key, run_agent, check, grade, requires_key
from kata_content import BY_SLUG

KATA = BY_SLUG["evaluation"]
load_env()
print("API key configured:" , has_api_key())


In [ ]:
# ✏️ Your code — fill in the TODOs
from google.genai import types
from google.adk.evaluation.eval_set import EvalSet
from google.adk.evaluation.eval_case import EvalCase, Invocation, IntermediateData

# TODO: build an Invocation for "What's the weather in Tokyo?" whose
#       intermediate_data.tool_uses = [types.FunctionCall(name=..., args=...)]
invocation = None

# TODO: wrap it in an EvalCase(eval_id="weather_tokyo") and an EvalSet.
eval_set = None

In [ ]:
# ✅ Check your work (offline — grades the symbols you defined above)
results = KATA.run_checks(globals())
grade([check(r.label, r.passed, r.detail) for r in results])


In [ ]:
# ▶️ Run it live (needs GOOGLE_API_KEY). Each run is a fresh single-turn session.
agent = globals().get(KATA.chat_symbol)

if not KATA.chattable:
    print("This kata has no chat agent — the Check cell is the goal. 🎯")
elif has_api_key() and agent is not None:
    result = await run_agent(agent, 'Hello!')   # noqa: F704  (top-level await is fine in Jupyter)
    print("Response:", result.text)
    if result.tool_calls:
        print("Tools called:", result.tool_calls, result.tool_args)
    if result.transfers:
        print("Transferred to:", result.transfers)
    if result.state:
        print("Session state:", result.state)
else:
    requires_key(lambda: None)


---
### Stuck? Reveal the reference solution

<details>
<summary>Show solution</summary>

```python
from google.genai import types
from google.adk.evaluation.eval_set import EvalSet
from google.adk.evaluation.eval_case import EvalCase, Invocation, IntermediateData

invocation = Invocation(
    invocation_id="weather_tokyo_1",
    user_content=types.Content(role="user", parts=[types.Part(text="What's the weather in Tokyo?")]),
    final_response=types.Content(role="model", parts=[types.Part(text="The weather in Tokyo is clear, 28 °C.")]),
    intermediate_data=IntermediateData(
        tool_uses=[types.FunctionCall(name="get_weather", args={"city": "Tokyo"})]
    ),
)
eval_set = EvalSet(
    eval_set_id="weather_basics",
    name="weather basics",
    eval_cases=[EvalCase(eval_id="weather_tokyo", conversation=[invocation])],
)
```

</details>

When you're done, try the same kata in the React app's live chat (`./dev.sh`
from the repo root) to watch the tool-call traces.
